# Final pipeline for risk and protective factor analysis using the discovery and retrospective DIOPTRA data

## Handling and organizing the data from DIOPTRA's dashboard

### Function for retrieving DIOPTRA project's data from Postman 

In [ ]:
import requests
import pandas as pd

def fetch_all_records(base_url, token, page_size=5000, clinic='chul'):
    headers = { "Authorization": f"Bearer {token}"}
    path_info = None
    dfs = {}
    count = 0
    total_records = 0
    max_records = 1
    new_url = base_url
    if page_size != 5000:
        new_url = new_url.replace("page_size=5000", "page_size={}".format(page_size))
    if clinic != 'chul':
        new_url = new_url.replace("clinical_site=chul", "clinical_site={}".format(clinic))

    print(new_url)

    while total_records < max_records:
        if path_info is None:
            url = new_url
        else:
            new_path = ",".join(str(x) for x in path_info)
            url = "{}&search_after={}".format(new_url, new_path)
        
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Request failed with status code: {response.status_code}")
            print(f"Response content: {response.text}")
            break

        data = response.json()
        
        # Check if results list is empty
        if not data.get('results') or len(data['results']) == 0:
            print(f"✗ Error: API returned empty results")
            print(f"API Response: {data}")
            break
        
        records = data['results'][0]['results']
        max_records = data['results'][0]['total_hits']
        print(max_records)

        remaining_records = max_records - total_records
        records = records[:remaining_records]

        df = pd.DataFrame(records)
        dfs[f"batch_{count+1}"] = df
        

        total_records += len(records)

        if total_records >= max_records:
            print(f"Reached {max_records} records. Stopping.")
            break

        path_info = data['results'][0]['search_after']
        if not path_info:
            print("No more pages to fetch.")
            break

        count += 1
        print(f"Fetched batch {count}, total records: {total_records}, next search_after: {path_info}")

    final_df = pd.concat(dfs.values(), ignore_index=True)
    print(final_df)
    return dfs, final_df, data

print("Function defined. Ready to fetch data from all clinical sites!")

In [ ]:
# # Install Google Drive API libraries (run this once)
# import subprocess
# import sys

# print("Installing Google Drive API libraries...")
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-auth-oauthlib", "google-auth-httplib2", "google-api-python-client"])
# print("✓ Libraries installed!")

### Saving automatically all the clinical site's excel files to a shared (with my team) Google Drive file using google-API.

Nothing saved locally, so my colleagues can both run the same notebook without errors and explore the data from the shared Google Drive.

In [ ]:
# Google Drive Authentication and Setup (Desktop OAuth 2.0)
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from googleapiclient.errors import HttpError
import pickle
import os
import time
import io
import tempfile
import requests
import json

# ===== CONFIGURATION: Google Drive File IDs for Shared Credentials =====
# File IDs from your shared Google Drive folder
CREDENTIALS_FILE_ID = "FILL IN THE GOOGLE DRIVE API CREDENTIALS FILE ID"  # dioptra_data_credentials_API_google_drive.json 
TOKEN_FILE_ID = "FILL IN THE GOOGLE DRIVE API TOKEN FILE ID"  # token.pickle

# Use temporary directory for all files
temp_dir = tempfile.gettempdir()
credentials_path = os.path.join(temp_dir, "dioptra_data_credentials_API_google_drive.json")
token_path = os.path.join(temp_dir, "token.pickle")

# Function to download file from Google Drive using direct URL (no auth needed for public files)
def download_file_from_google_drive_direct(file_id, local_path):
    """Download a public file from Google Drive using direct download link"""
    try:
        url = f"https://drive.google.com/uc?export=download&id={file_id}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            f.write(response.content)
        print(f"✓ Downloaded: {local_path}")
        return True
    except Exception as error:
        print(f'✗ Failed to download: {error}')
        return False

# Download credentials file if not already present
if not os.path.exists(credentials_path) or os.path.getsize(credentials_path) == 0:
    print("Downloading credentials file from shared Google Drive...")
    if not download_file_from_google_drive_direct(CREDENTIALS_FILE_ID, credentials_path):
        raise RuntimeError("Failed to download credentials file. Make sure it's shared publicly on Google Drive.")
else:
    print(f"✓ Credentials file already cached locally")

# Authenticate using desktop OAuth 2.0
try:
    scopes = ['https://www.googleapis.com/auth/drive']
    
    # Try to load existing token
    if os.path.exists(token_path):
        with open(token_path, 'rb') as token_file:
            credentials = pickle.load(token_file)
            print("✓ Loaded existing token from local cache")
    else:
        # Create new authorization flow using downloaded credentials file
        flow = InstalledAppFlow.from_client_secrets_file(credentials_path, scopes=scopes)
        credentials = flow.run_local_server(port=0)
        
        # Save token for future use
        with open(token_path, 'wb') as token_file:
            pickle.dump(credentials, token_file)
        print("✓ Created new token and saved to local cache")
    
    drive_service = build('drive', 'v3', credentials=credentials)
    print("✓ Successfully authenticated with Google Drive!")
except Exception as e:
    print(f"✗ Authentication failed: {e}")
    raise

# Function to download file from Google Drive using Drive API
def download_from_google_drive(service, file_id, local_path):
    """Download a file from Google Drive by file ID using Drive API"""
    try:
        request = service.files().get_media(fileId=file_id)
        fh = io.FileIO(local_path, 'wb')
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while done is False:
            status, done = downloader.next_chunk()
        fh.close()
        print(f"✓ Downloaded: {local_path}")
        return True
    except HttpError as error:
        print(f'✗ An error occurred: {error}')
        return False

# Download shared token from Google Drive (optional - for sharing across machines)
print("\n" + "="*60)
print("CHECKING SHARED TOKEN FROM GOOGLE DRIVE")
print("="*60)
try:
    if not os.path.exists(token_path) or os.path.getsize(token_path) == 0:
        print("Downloading token file from shared Google Drive...")
        download_from_google_drive(drive_service, TOKEN_FILE_ID, token_path)
    else:
        print(f"✓ Token file already cached locally")
except Exception as e:
    print(f"⚠ Could not check/download shared token: {e}")
print("="*60)

# Function to find folder by name in Google Drive
def find_folder_by_name(service, folder_name):
    try:
        results = service.files().list(
            q=f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
            spaces='drive',
            fields='files(id, name)',
            pageSize=10
        ).execute()
        
        folders = results.get('files', [])
        if folders:
            return folders[0]['id']
        else:
            print(f"✗ Folder '{folder_name}' not found in Google Drive")
            return None
    except HttpError as error:
        print(f'✗ An error occurred: {error}')
        return None

# Find the dioptra_project folder
dioptra_folder_id = find_folder_by_name(drive_service, 'dioptra_project')
if dioptra_folder_id:
    print(f"✓ Found dioptra_project folder (ID: {dioptra_folder_id})")

# Function to find or create a folder in Google Drive
def find_or_create_folder(service, folder_name, parent_folder_id):
    """Find existing folder or create a new one in the specified parent folder"""
    try:
        # First, try to find the folder
        results = service.files().list(
            q=f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_folder_id}' in parents and trashed=false",
            spaces='drive',
            fields='files(id, name)',
            pageSize=10
        ).execute()
        
        folders = results.get('files', [])
        if folders:
            print(f"✓ Found existing folder '{folder_name}'")
            return folders[0]['id']
        else:
            # Create the folder if it doesn't exist
            file_metadata = {
                'name': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [parent_folder_id]
            }
            folder = service.files().create(body=file_metadata, fields='id').execute()
            new_folder_id = folder.get('id')
            print(f"✓ Created new folder '{folder_name}' (ID: {new_folder_id})")
            return new_folder_id
    except HttpError as error:
        print(f'✗ An error occurred: {error}')
        return None

# Find the DATA folder inside dioptra_project
data_folder_id = find_folder_by_name(drive_service, 'DATA')
if data_folder_id:
    print(f"✓ Found DATA folder (ID: {data_folder_id})")
else:
    print(f"✗ DATA folder not found in Google Drive - creating it")
    if dioptra_folder_id:
        data_folder_id = find_or_create_folder(drive_service, 'DATA', dioptra_folder_id)

# Function to upload file to Google Drive folder with retry logic
def upload_to_google_drive(service, file_path, folder_id, file_name, max_retries=3):
    for attempt in range(max_retries):
        try:
            file_metadata = {
                'name': file_name,
                'parents': [folder_id]
            }
            media = MediaFileUpload(file_path, mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', resumable=True)
            file = service.files().create(body=file_metadata, media_body=media, fields='id').execute()
            return file.get('id')
        except Exception as error:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f'  ⚠ Upload attempt {attempt + 1} failed: {type(error).__name__}. Retrying in {wait_time}s...')
                time.sleep(wait_time)
            else:
                print(f'✗ Upload failed after {max_retries} attempts: {error}')
                return None

print("✓ Google Drive functions ready!")

### Using postman token and fetch_all_records function to retrieve the desired clinical site datasets. 

Also, using google-API all the files are saved automatically with the today date inside a file having the name 'today_date'_data.

* Below, it is necessary to take the postman token and paste it on token = "..."

In [ ]:
# List of all clinical sites
clinical_sites = ['agsavvas', 'burgos', 'chul', 'goc', 'graz', 'nkua', 'rsyd', 'ukcm', 'kbd']
# clinical_sites = ['agsavvas'] # If you want to save specific clinical site data only adjust this list to include the desired clinical site(s)

# Base URL template
base_url_template = "FILL IN THE DIOPTRA ENDPOINT URL"

# Postman Token (same for all requests)
token = "FILL IN THE POSTMAN TOKEN"

# Fetch data for all clinical sites and upload to Google Drive
from datetime import datetime
import tempfile
import os

# Create dated folder name: %d_%m_%y_data (e.g., 01_04_26_data for April 1, 2026)
today_date_short = datetime.now().strftime('%d_%m_%y')
dated_folder_name = f"{today_date_short}_data"

# Create or find the dated folder inside the DATA folder
dated_folder_id = None
if data_folder_id:
    dated_folder_id = find_or_create_folder(drive_service, dated_folder_name, data_folder_id)
    print(f"✓ Using dated folder for data storage: {dated_folder_name}")
else:
    print("✗ Cannot create dated folder - DATA folder not found")

# Legacy date format for filename (just day_month)
today_date = datetime.now().strftime('%d_%m')

all_results = {}

for site in clinical_sites:
    print(f"\n{'='*60}")
    print(f"Fetching data for clinical site: {site}")
    print(f"{'='*60}")
    
    # Create URL for this site
    base_url = base_url_template.format(site)
    
    # Fetch data
    dfs, final_df, data = fetch_all_records(base_url=base_url, token=token)
    
    # Create temporary Excel file
    filename = f'{site}_data_{today_date}.xlsx'
    temp_excel_path = os.path.join(tempfile.gettempdir(), filename)
    final_df.to_excel(temp_excel_path, index=False)
    print(f"✓ Created temporary file: {filename}")
    
    # Upload to Google Drive - now using the dated folder inside DATA
    if dated_folder_id:
        file_id = upload_to_google_drive(drive_service, temp_excel_path, dated_folder_id, filename)
        if file_id:
            print(f"✓ Uploaded to Google Drive: {filename}")
            print(f"  Location: DATA/{dated_folder_name}/{filename}")
        else:
            print(f"✗ Failed to upload: {filename}")
    else:
        print(f"✗ Cannot upload - dated folder not found in DATA directory")
    
    # Clean up temporary file
    try:
        os.remove(temp_excel_path)
    except:
        pass
    
    # Store results
    all_results[site] = {
        'dataframe': final_df,
        'batches': dfs,
        'filename': filename,
        'record_count': len(final_df),
        'uploaded': file_id is not None if dated_folder_id else False
    }

print(f"\n{'='*60}")
print("SUMMARY - All files uploaded to Google Drive!")
print(f"{'='*60}")
for site, result in all_results.items():
    status = "✓ Uploaded" if result['uploaded'] else "✗ Failed"
    print(f"{site}: {result['record_count']} records → {result['filename']} ({status})")

In [ ]:
# Example of the structure of the all_results dictionary:
# all_results['burgos']['dataframe'] # This gives the complete dataframe for the 'burgos'
# all_results['burgos']['batches'] # This gives the dictionary of batches for 'burgos' (if there were multiple batches)
# all_results['burgos']['filename'] # This gives the filename where 'burgos' data is saved
# all_results['burgos']['record_count'] # This gives the total number of records fetched for 'burgos'
# all_results['burgos']['uploaded'] # This indicates if the file was successfully uploaded to Google Drive

### Creating DataFrames for each clinical sites and exploring basic info

In [ ]:
# Load each clinical site's data into a DataFrame using the all_results dictionary
df_agsavvas = all_results['agsavvas']['dataframe']
df_burgos = all_results['burgos']['dataframe']
df_chul = all_results['chul']['dataframe']
df_goc = all_results['goc']['dataframe']
df_graz = all_results['graz']['dataframe']
df_kbd = all_results['kbd']['dataframe']
df_nkua = all_results['nkua']['dataframe']
df_rsyd = all_results['rsyd']['dataframe']
df_ukcm = all_results['ukcm']['dataframe']

In [ ]:
# The total numbers (with duplicates) with dioptra_study=1 per clinical_site
clinical_sites = [df_agsavvas, df_burgos, df_chul, df_goc, df_graz, df_kbd, df_nkua, df_rsyd, df_ukcm]
site_names = ["agsavvas", "burgos", "chul", "goc", "graz", "kbd", "nkua", "rsyd", "ukcm"]

for name, site in zip(site_names, clinical_sites):
    print(f"{name}: {(site['dioptra_study'] == 1).sum()}" )

In [ ]:
# The total numbers (with duplicates) with dioptra_study=2 per clinical_site
clinical_sites = [df_agsavvas, df_burgos, df_chul, df_goc, df_graz, df_kbd, df_nkua, df_rsyd, df_ukcm]
site_names = ["agsavvas", "burgos", "chul", "goc", "graz", "kbd", "nkua", "rsyd", "ukcm"]

for name, site in zip(site_names, clinical_sites):
    print(f"{name}: {(site['dioptra_study'] == 2).sum()}" )

In [ ]:
# The total numbers (with duplicates) with dioptra_study=3 per clinical_site
clinical_sites = [df_agsavvas, df_burgos, df_chul, df_goc, df_graz, df_kbd, df_nkua, df_rsyd, df_ukcm]
site_names = ["agsavvas", "burgos", "chul", "goc", "graz", "kbd", "nkua", "rsyd", "ukcm"]

for name, site in zip(site_names, clinical_sites):
    print(f"{name}: {(site['dioptra_study'] == 3).sum()}" )

In [ ]:
# The total numbers (with duplicates) with dioptra_study=4 per clinical_site
clinical_sites = [df_agsavvas, df_burgos, df_chul, df_goc, df_graz, df_kbd, df_nkua, df_rsyd, df_ukcm]
site_names = ["agsavvas", "burgos", "chul", "goc", "graz", "kbd", "nkua", "rsyd", "ukcm"]

for name, site in zip(site_names, clinical_sites):
    print(f"{name}: {(site['dioptra_study'] == 4).sum()}" )

## Data cleaning per clinical site

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#linking google drives excel files
import requests 
from io import BytesIO

import itertools

### Helper functions

#### Reading excel from Google Drive

In [ ]:
def load_excel_from_drive(file_id):
    """
    Load an Excel file directly from Google Drive into a pandas DataFrame
    without saving it locally.
    
    """
    
    # Create direct download URL
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    
    # Request file content
    response = requests.get(url)
    
    # Check for permission / download errors
    try:
        response.raise_for_status()
    except Exception as e:
        raise Exception("Error downloading file. Check permissions or file ID.") from e
    
    # Load Excel into DataFrame directly from memory
    try:
        df = pd.read_excel(BytesIO(response.content))
    except Exception as e:
        raise Exception("Downloaded content is not a valid Excel file.") from e
    
    return df

#### Filling the dioptra_diagnosis column

In [ ]:
def fill_group_diagnosis_study_12(df):
    # Work only on rows where dioptra_study = 1 or 2
    mask = df['dioptra_study'].isin([1,2])
    df_sub = df.loc[mask].copy()

    # Columns used to determine dioptra_group_diagnosis
    diag_cols = [
        'diagnosis_healthy',
        'diagnosis_non_advanced_adenomas',
        'diagnosis_advanced_adenomas',
        'diagnosis_crc'
    ]
     
    # This is only for NKUΑ having 'No' values on diag_cols
    df_sub[diag_cols] = df_sub[diag_cols].replace( [ 'No', 'NO', 'Νο', 'ΝΟ'], pd.NA )    # The first No and NO are latins while the second ones are greeks

    # This is only for RSYD having '.' values on diag_cols
    df_sub[diag_cols] = df_sub[diag_cols].replace('.', pd.NA)

    # Drop rows where ALL 4 diagnosis columns are missing
    df_sub = df_sub.dropna(subset = diag_cols, how = 'all')


    # Fill dioptra_group_diagnosis based on which diagnosis col is non-null
    df_sub.loc[df_sub['diagnosis_healthy'].notna(), 'dioptra_group_diagnosis'] = 1
    df_sub.loc[df_sub['diagnosis_non_advanced_adenomas'].notna(), 'dioptra_group_diagnosis'] = 2
    df_sub.loc[df_sub['diagnosis_advanced_adenomas'].notna(), 'dioptra_group_diagnosis'] = 3
    df_sub.loc[df_sub['diagnosis_crc'].notna(), 'dioptra_group_diagnosis'] = 4

    # Put the updated rows back into the original dataframe
    df.loc[df_sub.index] = df_sub
    df = df.drop(df.loc[mask].index.difference(df_sub.index))
    
    return df

#### Handling the duplicates

In [ ]:
# This function takes as a group the sub-DataFrame generating below from .groupby('dioptra_id') and returns a one-row DataFrame with the characteristics described into the function.
# It picks the unique row depending every time on the dioptra_study
def pick_row(group):

    # If there are duplicated rows with dioptra_study = 3 & 4 then keep only these ones with dioptra_study = 4
    if set(group['dioptra_study']) == {3, 4}:
        group = group[group['dioptra_study'] == 4]

    # Here we check only the duplicates with dioptra_study = 1, 2. Here the column 'sample_diagnosis_group' does not exist. The 'colon_time_point' exists here but each diagnosis on each colon_time point is representative of the other columns,
    # so we do not need to be focused on the diagnoses at all the colon_time_points. Finally, we pick the most severe case in order to try increasing the crc cases!!
    if group['dioptra_study'].isin([1,2]).all(): # handles rows either only with ds=1, or only ds=2, or combination like ds = 1,2

        if group['dioptra_group_diagnosis'].nunique() == 1:
            missing_counts = group.isna().sum(axis=1) # Count missing values per row (True=missing so we take .sum())
            index = missing_counts.idxmin()   # pick row with fewest missing values   
        else:
            max_diag = group['dioptra_group_diagnosis'].max() # find the max diagnosis
            subset = group[group['dioptra_group_diagnosis'] == max_diag] # keep all the rows having the max diagnosis
            missing_counts = subset.isna().sum(axis=1) # Count missing values per row (True=missing so we take .sum())
            index = missing_counts.idxmin() # pick row with fewest missing values 

    # Here we check the duplicates with dioptra_study 3 or 4. 
    else:

        if group['dioptra_group_diagnosis'].nunique() == 1 and group['sample_diagnosis_group'].nunique() == 1:
            missing_counts = group.isna().sum(axis=1) # Count missing values per row (True=missing so we take .sum())
            index = missing_counts.idxmin()   # pick row with fewest missing values        

        elif group['dioptra_group_diagnosis'].nunique() == 1:
            index = group['sample_diagnosis_group'].idxmax()

        else:
            index = group['dioptra_group_diagnosis'].idxmax()

    return group.loc[[index]]  

#### Check when there are IDs, with ds = (1 or 2), having multiple diagnosis (See UKCM)

In [ ]:
def filter_rows_by_diagnosis_completeness(df):
    """
    Returns a dataframe containing rows where the diagnosis columns
    have between 2 and 4 non-missing values.
    """

    diag_cols = [
        'diagnosis_healthy',
        'diagnosis_non_advanced_adenomas',
        'diagnosis_advanced_adenomas',
        'diagnosis_crc'
    ]

    # Count non-missing values in diagnosis columns
    non_missing_count = df[diag_cols].notna().sum(axis=1)

    # Keep rows with 2, 3, or 4 non-missing diagnosis values
    df_selected = df[(non_missing_count >= 2) & (non_missing_count <= 4)].copy()

    # Keep only the required columns
    cols_to_keep = ['dioptra_id', 'dioptra_study'] + diag_cols
    df_selected = df_selected[cols_to_keep]
    
    return df_selected


#### Finding the best combination among all clinical sites

In [ ]:
def analyze_best_dataframe_combination(df_list):
    """
    Takes a list of DataFrames and finds:
    1. Common columns across ALL dataframes
    2. The best combination of ANY SIZE (2..n)
       that has the highest number of common columns.
    Returns:
       - common columns across all
       - best combination (tuple of indices)
       - common columns in that combination
    """

    # Common columns across ALL dataframes
    common_cols_all = set(df_list[0].columns)
    for df in df_list[1:]:
        common_cols_all &= set(df.columns)
    common_cols_all = sorted(common_cols_all)

    print(f"Common columns across ALL dataframes:\n{common_cols_all}\nLength of those columns:\n{len(common_cols_all)}")

    
    # Find best combination of ANY size
    best_combo = None
    best_common_cols = []
    max_common_count = -1

    n = len(df_list)

    # Evaluate all combination sizes from 2 → n
    for r in range(2, n + 1):
        for combo in itertools.combinations(range(n), r):
            # Start with columns of the first df in combo
            common_cols = set(df_list[combo[0]].columns)

            # Intersect with rest of combination
            for idx in combo[1:]:
                common_cols &= set(df_list[idx].columns)

            # Check if this combination is better
            if len(common_cols) > max_common_count:
                max_common_count = len(common_cols)
                best_combo = combo
                best_common_cols = sorted(list(common_cols))

    print("Best dataframe combination regarding the highest number of columns:")
    print(f"DataFrame indices: {best_combo}")
    print(f"Number of common columns: {max_common_count}")
    print("Common columns:")
    print(best_common_cols)

    return common_cols_all, best_combo, best_common_cols

### AGSAVVAS

In [ ]:
df_agsavvas_1_2 = df_agsavvas[(df_agsavvas['dioptra_study'] == 1) | (df_agsavvas['dioptra_study'] == 2)]
df_agsavvas_1_2

In [ ]:
df_agsavvas_1_2_fill = fill_group_diagnosis_study_12(df = df_agsavvas_1_2).reset_index(drop = True)
df_agsavvas_1_2_fill

In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_agsavvas_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
print(f"The missing values of the above DataFrame is {df_agsavvas_1_2_fill.isnull().sum().sum()} and the corresponding percentage is {((df_agsavvas_1_2_fill.isnull().sum().sum()/df_agsavvas_1_2_fill.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_agsavvas_1_2_fill.columns[ df_agsavvas_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_agsavvas_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_agsavvas_12_drop_columns = df_agsavvas_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_agsavvas_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_agsavvas = df_agsavvas_12_drop_columns[df_agsavvas_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_agsavvas

We do not have duplicates here!

In [ ]:
# Percentage of missing values for each column
(df_agsavvas_12_drop_columns.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_agsavvas_12_drop_columns.isnull().mean() * 100

# Select the columns which have missing percentage > 70%
col_drop = missing_percentage[missing_percentage > 70].index

# Drop the above columns from the DataFrame
df_agsavvas_clean_col = df_agsavvas_12_drop_columns.drop(columns = col_drop)

#print the df
df_agsavvas_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_agsavvas_clean_col, x = df_agsavvas_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_agsavvas_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_agsavvas_clean_col['dioptra_group_diagnosis'].value_counts()

### BURGOS

In [ ]:
df_burgos_1_2 = df_burgos[(df_burgos['dioptra_study'] == 1) | (df_burgos['dioptra_study'] == 2)]
df_burgos_1_2

In [ ]:
df_burgos_1_2_fill = fill_group_diagnosis_study_12(df = df_burgos_1_2).reset_index(drop = True)
df_burgos_1_2_fill

In [ ]:
df_burgos_selected = filter_rows_by_diagnosis_completeness(df = df_burgos_1_2_fill)
df_burgos_selected

In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_burgos_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
print(f"The missing values of the above DataFrame is {df_burgos_1_2_fill.isnull().sum().sum()} and the corresponding percentage is {((df_burgos_1_2_fill.isnull().sum().sum()/df_burgos_1_2_fill.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_burgos_1_2_fill.columns[ df_burgos_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_burgos_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_burgos_12_drop_columns = df_burgos_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_burgos_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_burgos = df_burgos_12_drop_columns[df_burgos_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_burgos

We do not have duplicates here!

In [ ]:
# Percentage of missing values for each column
(df_burgos_12_drop_columns.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_burgos_12_drop_columns.isnull().mean() * 100

# Select the columns which have missing percentage > 90%
col_drop = missing_percentage[missing_percentage > 90].index

# Drop the above columns from the DataFrame
df_burgos_clean_col = df_burgos_12_drop_columns.drop(columns = col_drop)

#print the df
df_burgos_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_burgos_clean_col, x = df_burgos_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_burgos_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_burgos_clean_col['dioptra_group_diagnosis'].value_counts()

### CHUL

In [ ]:
df_chul_1_2 = df_chul[(df_chul['dioptra_study'] == 1) | (df_chul['dioptra_study'] == 2)]
df_chul_1_2

In [ ]:
df_chul_1_2_fill = fill_group_diagnosis_study_12(df = df_chul_1_2).reset_index(drop = True)
df_chul_1_2_fill

In [ ]:
df_chul_selected = filter_rows_by_diagnosis_completeness(df = df_chul_1_2_fill)
df_chul_selected

In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_chul_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id' for the df_chul_1_2_fill DataFrame. This is applied only to find the percentage of the missing values immediately after dropping the duplicated values.
df_chul_1_2_fill_drop_duplicates = (df_chul_1_2_fill
                                    .groupby('dioptra_id', group_keys = False)
                                    .apply(pick_row)
                                    .reset_index(drop = True)
                                    )

df_chul_1_2_fill_drop_duplicates

In [ ]:
print(f"The missing values of the above DataFrame is {df_chul_1_2_fill_drop_duplicates.isnull().sum().sum()} and the corresponding percentage is {((df_chul_1_2_fill_drop_duplicates.isnull().sum().sum()/df_chul_1_2_fill_drop_duplicates.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_chul_1_2_fill.columns[ df_chul_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_chul_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_chul_12_drop_columns = df_chul_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_chul_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_chul = df_chul_12_drop_columns[df_chul_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_chul

In [ ]:
# For each duplicated dioptra_id value find all the columns having at least one different value
diff_chul_report = {}

# dupl_chul.groupby("dioptra_id") splits the df into groups by the dioptra_id column. So, id gives the diopta_id value and group gives the sub-df containing only the rows of the corresponding id
for id, group in dupl_chul.groupby("dioptra_id"):
    
    # return a Series whose index is the column names and whose values are the number of distinct entries for that column inside to the corresponding group
    varying_cols = group.nunique(dropna = False) 

    # only store if we actually found differences
    cols_with_diffs = varying_cols[varying_cols > 1].index.tolist()
    diff_chul_report[id] = cols_with_diffs

diff_chul_report

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id'
df_chul_clean_dupl = (df_chul_12_drop_columns
                       .groupby('dioptra_id', group_keys = False)
                       .apply(pick_row)
                       .reset_index(drop = True)
                      )

df_chul_clean_dupl

In [ ]:
print(f"The missing values of the above DataFrame is {df_chul_clean_dupl.isnull().sum().sum()} and the corresponding percentage is {((df_chul_clean_dupl.isnull().sum().sum()/df_chul_clean_dupl.size)*100):.2f}%")

In [ ]:
# Percentage of missing values for each column
(df_chul_clean_dupl.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_chul_clean_dupl.isnull().mean() * 100

# Select the columns which have missing percentage > 97%
col_drop = missing_percentage[missing_percentage > 97].index

# Drop the above columns from the DataFrame
df_chul_clean_col = df_chul_clean_dupl.drop(columns = col_drop)

#print the df
df_chul_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_chul_clean_col, x = df_chul_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_chul_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_chul_clean_col['dioptra_group_diagnosis'].value_counts()

### GOC

In [ ]:
df_goc_1_2 = df_goc[(df_goc['dioptra_study'] == 1) | (df_goc['dioptra_study'] == 2)]
df_goc_1_2

In [ ]:
df_goc_1_2_fill = fill_group_diagnosis_study_12(df = df_goc_1_2).reset_index(drop = True)
df_goc_1_2_fill

In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_goc_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
print(f"The missing values of the above DataFrame is {df_goc_1_2_fill.isnull().sum().sum()} and the corresponding percentage is {((df_goc_1_2_fill.isnull().sum().sum()/df_goc_1_2_fill.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_goc_1_2_fill.columns[ df_goc_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_goc_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_goc_12_drop_columns = df_goc_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_goc_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_goc = df_goc_12_drop_columns[df_goc_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_goc

Here we do not have duplicates

In [ ]:
# Percentage of missing values for each column
(df_goc_12_drop_columns.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_goc_12_drop_columns.isnull().mean() * 100

# Select the columns which have missing percentage > 70%
col_drop = missing_percentage[missing_percentage > 90].index

# Drop the above columns from the DataFrame
df_goc_clean_col = df_goc_12_drop_columns.drop(columns = col_drop)

#print the df
df_goc_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_goc_clean_col, x = df_goc_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_goc_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_goc_clean_col['dioptra_group_diagnosis'].value_counts()

### GRAZ

In [ ]:
df_graz_1_2 = df_graz[(df_graz['dioptra_study'] == 1) | (df_graz['dioptra_study'] == 2)]
df_graz_1_2

In [ ]:
df_graz_1_2_fill = fill_group_diagnosis_study_12(df = df_graz_1_2).reset_index(drop = True)
df_graz_1_2_fill

In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_graz_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id' for the df_graz_1_2_fill DataFrame. This is applied only to find the percentage of the missing values immediately after dropping the duplicated values.
df_graz_1_2_fill_drop_duplicates = (df_graz_1_2_fill
                                    .groupby('dioptra_id', group_keys = False)
                                    .apply(pick_row)
                                    .reset_index(drop = True)
                                    )

df_graz_1_2_fill_drop_duplicates

In [ ]:
print(f"The missing values of the above DataFrame is {df_graz_1_2_fill_drop_duplicates.isnull().sum().sum()} and the corresponding percentage is {((df_graz_1_2_fill_drop_duplicates.isnull().sum().sum()/df_graz_1_2_fill_drop_duplicates.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_graz_1_2_fill.columns[ df_graz_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_graz_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_graz_12_drop_columns = df_graz_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_graz_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_graz = df_graz_12_drop_columns[df_graz_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_graz

In [ ]:
# For each duplicated dioptra_id value find all the columns having at least one different value
diff_graz_report = {}

# dupl_graz.groupby("dioptra_id") splits the df into groups by the dioptra_id column. So, id gives the diopta_id value and group gives the sub-df containing only the rows of the corresponding id
for id, group in dupl_graz.groupby("dioptra_id"):
    
    # return a Series whose index is the column names and whose values are the number of distinct entries for that column inside to the corresponding group
    varying_cols = group.nunique(dropna = False) 

    # only store if we actually found differences
    cols_with_diffs = varying_cols[varying_cols > 1].index.tolist()
    diff_graz_report[id] = cols_with_diffs

diff_graz_report

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id'
df_graz_clean_dupl = (df_graz_12_drop_columns
                       .groupby('dioptra_id', group_keys = False)
                       .apply(pick_row)
                       .reset_index(drop = True)
                      )

df_graz_clean_dupl

In [ ]:
print(f"The missing values of the above DataFrame is {df_graz_clean_dupl.isnull().sum().sum()} and the corresponding percentage is {((df_graz_clean_dupl.isnull().sum().sum()/df_graz_clean_dupl.size)*100):.2f}%")

In [ ]:
# Percentage of missing values for each column
(df_graz_clean_dupl.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_graz_clean_dupl.isnull().mean() * 100

# Select the columns which have missing percentage > 70%
col_drop = missing_percentage[missing_percentage > 90].index

# Drop the above columns from the DataFrame
df_graz_clean_col = df_graz_clean_dupl.drop(columns = col_drop)

#print the df
df_graz_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_graz_clean_col, x = df_graz_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_graz_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_graz_clean_col['dioptra_group_diagnosis'].value_counts()

### NKUA

In [ ]:
df_nkua_1_2 = df_nkua[(df_nkua['dioptra_study'] == 1) | (df_nkua['dioptra_study'] == 2)]
df_nkua_1_2

In [ ]:
df_nkua_1_2_fill = fill_group_diagnosis_study_12(df = df_nkua_1_2).reset_index(drop = True)
df_nkua_1_2_fill

In [ ]:
df_nkua_1_2_fill['age_colon_time_point'] = df_nkua_1_2_fill['age_colon_time_point'].fillna(df_nkua_1_2_fill['enrollment_age'])


In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_nkua_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id' for the df_nkua_1_2_fill DataFrame. This is applied only to find the percentage of the missing values immediately after dropping the duplicated values.
df_nkua_1_2_fill_drop_duplicates = (df_nkua_1_2_fill
                                    .groupby('dioptra_id', group_keys = False)
                                    .apply(pick_row)
                                    .reset_index(drop = True)
                                    )

df_nkua_1_2_fill_drop_duplicates

In [ ]:
print(f"The missing values of the above DataFrame is {df_nkua_1_2_fill_drop_duplicates.isnull().sum().sum()} and the corresponding percentage is {((df_nkua_1_2_fill_drop_duplicates.isnull().sum().sum()/df_nkua_1_2_fill_drop_duplicates.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_nkua_1_2_fill.columns[ df_nkua_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_nkua_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_nkua_12_drop_columns = df_nkua_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_nkua_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_nkua = df_nkua_12_drop_columns[df_nkua_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_nkua

In [ ]:
# For each duplicated dioptra_id value find all the columns having at least one different value
diff_nkua_report = {}

# dupl_nkua.groupby("dioptra_id") splits the df into groups by the dioptra_id column. So, id gives the diopta_id value and group gives the sub-df containing only the rows of the corresponding id
for id, group in dupl_nkua.groupby("dioptra_id"):
    
    # return a Series whose index is the column names and whose values are the number of distinct entries for that column inside to the corresponding group
    varying_cols = group.nunique(dropna = False) 

    # only store if we actually found differences
    cols_with_diffs = varying_cols[varying_cols > 1].index.tolist()
    diff_nkua_report[id] = cols_with_diffs

diff_nkua_report

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id'
df_nkua_clean_dupl = (df_nkua_12_drop_columns
                       .groupby('dioptra_id', group_keys = False)
                       .apply(pick_row)
                       .reset_index(drop = True)
                      )

df_nkua_clean_dupl

In [ ]:
print(f"The missing values of the above DataFrame is {df_nkua_clean_dupl.isnull().sum().sum()} and the corresponding percentage is {((df_nkua_clean_dupl.isnull().sum().sum()/df_nkua_clean_dupl.size)*100):.2f}%")

In [ ]:
# Fill some BMI values using the excel that NKUA provided
id_list = [
    "CP-01-0019", "CP-01-0022", "CP-01-0026", "CP-01-0031", "CP-01-0109",
    "CP-01-0112", "CP-01-0123", "CP-01-0142", "CP-01-0161", "CP-01-0162",
    "CP-01-0164", "CP-01-0187", "CP-01-0222", "CP-01-0224", "CP-01-0244",
    "CP-01-0261", "CP-01-0263", "CP-01-0267", "CP-01-0295", "CP-01-0310",
    "CP-01-0313", "CP-01-0324", "CP-01-0333", "CP-01-0340", "CP-01-0355",
    "CP-01-0357", "CP-01-0480", "CP-01-463"
]
bmi_list = [
    26.2, 32, 24.9, 21.3, 37.2,
    28.5, 24.4, 27.4, 25.7, 26.7,
    24.6, 27, 25.1, 24.4, 26.4,
    28.7, 20.5, 24.5, 28.7, 20.6,
    20.2, 25.1, 26.8, 57.4, 30.7,
    29.6, 20.2, 20.5
]

# This generates {""CP-01-0019": 26.2, "CP-01-0022": 32, ..."}
bmi_map = dict(zip(id_list, bmi_list))

# fill these missing BMI values for the IDs containing in the id_list
df_nkua_clean_dupl['bmi'] = df_nkua_clean_dupl.apply(
    lambda row: bmi_map.get(row['dioptra_id'], row['bmi']),
    axis=1
)


In [ ]:
# Furthermore NKUA informed us that the ID CP-01-0487 has dioptra_study=3 and not ds=1 as it is here. So we drop this row for the analysis since we have here only ds=1 or 2
df_nkua_clean_dupl = df_nkua_clean_dupl.drop(df_nkua_clean_dupl.index[df_nkua_clean_dupl['dioptra_id'] == "CP-01-0487"])

In [ ]:
# Percentage of missing values for each column
(df_nkua_clean_dupl.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_nkua_clean_dupl.isnull().mean() * 100

# Select the columns which have missing percentage > 80%
col_drop = missing_percentage[missing_percentage > 80].index # If we put here threshold 90% in order to keep the BMI column, then the final merged df is getting messy! So, we decide to not include the BMI in our analysis so far

# Drop the above columns from the DataFrame
df_nkua_clean_col = df_nkua_clean_dupl.drop(columns = col_drop)

#print the df
df_nkua_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_nkua_clean_col, x = df_nkua_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_nkua_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_nkua_clean_col['dioptra_group_diagnosis'].value_counts()

### RSYD

In [ ]:
df_rsyd_1_2 = df_rsyd[(df_rsyd['dioptra_study'] == 1) | (df_rsyd['dioptra_study'] == 2)]
df_rsyd_1_2

In [ ]:
df_rsyd_1_2_fill = fill_group_diagnosis_study_12(df = df_rsyd_1_2).reset_index(drop = True)
df_rsyd_1_2_fill

In [ ]:
df_rsyd_selected = filter_rows_by_diagnosis_completeness(df = df_rsyd_1_2_fill)
df_rsyd_selected

In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_rsyd_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id' for the df_rsyd_1_2_fill DataFrame. This is applied only to find the percentage of the missing values immediately after dropping the duplicated values.
df_rsyd_1_2_fill_drop_duplicates = (df_rsyd_1_2_fill
                                    .groupby('dioptra_id', group_keys = False)
                                    .apply(pick_row)
                                    .reset_index(drop = True)
                                    )

df_rsyd_1_2_fill_drop_duplicates

In [ ]:
print(f"The missing values of the above DataFrame is {df_rsyd_1_2_fill_drop_duplicates.isnull().sum().sum()} and the corresponding percentage is {((df_rsyd_1_2_fill_drop_duplicates.isnull().sum().sum()/df_rsyd_1_2_fill_drop_duplicates.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_rsyd_1_2_fill.columns[ df_rsyd_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_rsyd_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_rsyd_12_drop_columns = df_rsyd_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_rsyd_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_rsyd = df_rsyd_12_drop_columns[df_rsyd_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_rsyd

In [ ]:
# For each duplicated dioptra_id value find all the columns having at least one different value
diff_rsyd_report = {}

# dupl_rsyd.groupby("dioptra_id") splits the df into groups by the dioptra_id column. So, id gives the diopta_id value and group gives the sub-df containing only the rows of the corresponding id
for id, group in dupl_rsyd.groupby("dioptra_id"):
    
    # return a Series whose index is the column names and whose values are the number of distinct entries for that column inside to the corresponding group
    varying_cols = group.nunique(dropna = False) 

    # only store if we actually found differences
    cols_with_diffs = varying_cols[varying_cols > 1].index.tolist()
    diff_rsyd_report[id] = cols_with_diffs

diff_rsyd_report

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id'
df_rsyd_clean_dupl = (df_rsyd_12_drop_columns
                       .groupby('dioptra_id', group_keys = False)
                       .apply(pick_row)
                       .reset_index(drop = True)
                      )

df_rsyd_clean_dupl

In [ ]:
print(f"The missing values of the above DataFrame is {df_rsyd_clean_dupl.isnull().sum().sum()} and the corresponding percentage is {((df_rsyd_clean_dupl.isnull().sum().sum()/df_rsyd_clean_dupl.size)*100):.2f}%")

In [ ]:
# Percentage of missing values for each column
(df_rsyd_clean_dupl.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_rsyd_clean_dupl.isnull().mean() * 100

# Select the columns which have missing percentage > 70%
col_drop = missing_percentage[missing_percentage > 90].index

# Drop the above columns from the DataFrame
df_rsyd_clean_col = df_rsyd_clean_dupl.drop(columns = col_drop)

#print the df
df_rsyd_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_rsyd_clean_col, x = df_rsyd_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_rsyd_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_rsyd_clean_col['dioptra_group_diagnosis'].value_counts()

### UKCM

In [ ]:
df_ukcm_1_2 = df_ukcm[(df_ukcm['dioptra_study'] == 1) | (df_ukcm['dioptra_study'] == 2)]
df_ukcm_1_2

In [ ]:
df_ukcm_1_2_fill = fill_group_diagnosis_study_12(df = df_ukcm_1_2).reset_index(drop = True)
df_ukcm_1_2_fill

In [ ]:
df_ukcm_selected = filter_rows_by_diagnosis_completeness(df = df_ukcm_1_2_fill)
df_ukcm_selected

In [ ]:
# df_ukcm_selected.to_excel('diagnosis_confusion_ds=2_ukcm.xlsx', index = False)

In [ ]:
# Drop the columns 'diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc', because we have all the information on 'dioptra_group_diagnosis' column
df_ukcm_1_2_fill.drop(columns = ['diagnosis_healthy', 'diagnosis_non_advanced_adenomas', 'diagnosis_advanced_adenomas', 'diagnosis_crc'], inplace = True)

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id' for the df_ukcm_1_2_fill DataFrame. This is applied only to find the percentage of the missing values immediately after dropping the duplicated values.
df_ukcm_1_2_fill_drop_duplicates = (df_ukcm_1_2_fill
                                    .groupby('dioptra_id', group_keys = False)
                                    .apply(pick_row)
                                    .reset_index(drop = True)
                                    )

df_ukcm_1_2_fill_drop_duplicates

In [ ]:
print(f"The missing values of the above DataFrame is {df_ukcm_1_2_fill_drop_duplicates.isnull().sum().sum()} and the corresponding percentage is {((df_ukcm_1_2_fill_drop_duplicates.isnull().sum().sum()/df_ukcm_1_2_fill_drop_duplicates.size)*100):.2f}%")

In [ ]:
# Drop unnecessary columns
string_columns = ['country_res', 'weight', 'height', 'country_birth', 'country_lived', 'medication_other']

polyps_related_columns = ['time_period_v4_v278', 'neoplasm_malignant_history_category', 'last_fit_test_result', 'neoplasm_malignant_history', 'lynch_syndrome', 'positive_fit_test', 'polyposis_coli', 'colonoscopy_time_ellapsed', 'colonocopy_count',
                          'time_period_v4_v6', 'time_period_v4_v272', 'time_period_v4_v275', 'malignant_neoplasm_treatment', 'neoplasm_malignant_history_sub', 'charlson_comorbidity_index', 'colonoscopy_findings']

clinical_biology_until_the_end_columns = df_ukcm_1_2_fill.columns[ df_ukcm_1_2_fill.columns.get_loc("alanine_aminotransferase") : df_ukcm_1_2_fill.columns.get_loc("date") + 1 ].tolist()
if 'sample_diagnosis_group' in clinical_biology_until_the_end_columns:
    clinical_biology_until_the_end_columns.remove('sample_diagnosis_group')

df_ukcm_12_drop_columns = df_ukcm_1_2_fill.drop(columns = polyps_related_columns + clinical_biology_until_the_end_columns + string_columns)
df_ukcm_12_drop_columns

In [ ]:
# All the rows of duplicated dioptra_id values sorted by the dioptra_id column
dupl_ukcm = df_ukcm_12_drop_columns[df_ukcm_12_drop_columns['dioptra_id'].duplicated(keep = False)].sort_values(by = 'dioptra_id')
dupl_ukcm

In [ ]:
# For each duplicated dioptra_id value find all the columns having at least one different value
diff_ukcm_report = {}

# dupl_ukcm.groupby("dioptra_id") splits the df into groups by the dioptra_id column. So, id gives the diopta_id value and group gives the sub-df containing only the rows of the corresponding id
for id, group in dupl_ukcm.groupby("dioptra_id"):
    
    # return a Series whose index is the column names and whose values are the number of distinct entries for that column inside to the corresponding group
    varying_cols = group.nunique(dropna = False) 

    # only store if we actually found differences
    cols_with_diffs = varying_cols[varying_cols > 1].index.tolist()
    diff_ukcm_report[id] = cols_with_diffs

diff_ukcm_report

In [ ]:
# pick_row function implemented to drop the duplicated values grouped by 'dioptra_id'
df_ukcm_clean_dupl = (df_ukcm_12_drop_columns
                       .groupby('dioptra_id', group_keys = False)
                       .apply(pick_row)
                       .reset_index(drop = True)
                      )

df_ukcm_clean_dupl

In [ ]:
print(f"The missing values of the above DataFrame is {df_ukcm_clean_dupl.isnull().sum().sum()} and the corresponding percentage is {((df_ukcm_clean_dupl.isnull().sum().sum()/df_ukcm_clean_dupl.size)*100):.2f}%")

In [ ]:
# Percentage of missing values for each column
(df_ukcm_clean_dupl.isnull().mean().sort_values(ascending = False)*100).round(2)

In [ ]:
# The percentage of missing values of each column
missing_percentage = df_ukcm_clean_dupl.isnull().mean() * 100

# Select the columns which have missing percentage > 70%
col_drop = missing_percentage[missing_percentage > 90].index

# Drop the above columns from the DataFrame
df_ukcm_clean_col = df_ukcm_clean_dupl.drop(columns = col_drop)

#print the df
df_ukcm_clean_col

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
plt.figure(figsize = (12, 6))
sns.countplot(data = df_ukcm_clean_col, x = df_ukcm_clean_col['dioptra_group_diagnosis'].map(diagnosis_map), hue = 'dioptra_group_diagnosis', order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')

plt.xlabel('Diagnosis')
plt.ylabel('Number of records')
plt.title('Distribution of the df_ukcm_clean_col dataset regarding the (label) dioptra_group_diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
df_ukcm_clean_col['dioptra_group_diagnosis'].value_counts()

## Testing different merged datasets

In [ ]:
dfs_after_drop_duplicated_and_before_drop_columns = [df_agsavvas_1_2_fill, df_burgos_1_2_fill, df_chul_1_2_fill_drop_duplicates, df_goc_1_2_fill, df_graz_1_2_fill_drop_duplicates,
                                                     df_nkua_1_2_fill_drop_duplicates, df_rsyd_1_2_fill_drop_duplicates, df_ukcm_1_2_fill_drop_duplicates]

dfs = [df_agsavvas_clean_col, df_burgos_clean_col, df_chul_clean_col, df_goc_clean_col, df_graz_clean_col, df_nkua_clean_col, df_rsyd_clean_col, df_ukcm_clean_col]
dfs_burgos_goc_nkua = [df_burgos_clean_col, df_goc_clean_col, df_nkua_clean_col]
dfs_burgos_goc_nkua_chul = [df_burgos_clean_col, df_goc_clean_col, df_nkua_clean_col, df_chul_clean_col]

In [ ]:
common_cols_all, best_combo, best_common_cols = analyze_best_dataframe_combination(df_list = dfs)

In [ ]:
df_merged_full_columns_unique_ids = pd.concat(dfs_after_drop_duplicated_and_before_drop_columns, ignore_index = True, join = 'inner') # if some ids does not have some columns contained on others ids, join = 'outer' fill these columns with missing value, else if join = 'inner' only keep the common columns among all the dfs and drop the rest
df_merged_full_columns_unique_ids

In [ ]:
print(f"The missing values of the above DataFrame is {df_merged_full_columns_unique_ids.isnull().sum().sum()} and the corresponding percentage is {((df_merged_full_columns_unique_ids.isnull().sum().sum()/df_merged_full_columns_unique_ids.size)*100):.2f}%")

In [ ]:
df_merged_all = pd.concat(dfs, ignore_index = True, join = 'inner') # if some ids does not have some columns contained on others ids, join = 'outer' fill these columns with missing value, else if join = 'inner' only keep the common columns among all the dfs and drop the rest
df_merged_all

In [ ]:
common_cols_test = set(df_burgos_clean_col.columns) & set(df_goc_clean_col.columns) & set(df_nkua_clean_col.columns)
print(sorted(common_cols_test))
print(f"Common coloumns: {len(common_cols_test)}")

In [ ]:
common_cols_test2 = set(df_burgos_clean_col.columns) & set(df_goc_clean_col.columns)  & set(df_chul_clean_col.columns)
print(sorted(common_cols_test2))
print(f"Common coloumns: {len(common_cols_test2)}")

In [ ]:
common_cols_test3 = set(df_burgos_clean_col.columns) & set(df_goc_clean_col.columns)  & set(df_chul_clean_col.columns) & set(df_nkua_clean_col.columns)
print(sorted(common_cols_test3))
print(f"Common coloumns: {len(common_cols_test3)}")

In [ ]:
common_cols_test4 = set(df_burgos_clean_col.columns) & set(df_goc_clean_col.columns)  & set(df_chul_clean_col.columns) & set(df_nkua_clean_col.columns) & set(df_graz_clean_col.columns)
print(sorted(common_cols_test4))
print(f"Common coloumns: {len(common_cols_test4)}")

In [ ]:
# df_merged_all = pd.concat(dfs, ignore_index = True, join = 'outer') # join = 'inner' take as columns only the intersected columns common_cols
# df_merged_all

In [ ]:
df_merged_burgos_goc_nkua = pd.concat(dfs_burgos_goc_nkua, ignore_index = True, join = 'inner') # join = 'inner' take as columns only the intersected columns common_cols
df_merged_burgos_goc_nkua 

In [ ]:
df_merged_burgos_goc_nkua_chul = pd.concat(dfs_burgos_goc_nkua_chul, ignore_index = True, join = 'inner') # join= 'inner' take as columns only the intersected columns common_cols
df_merged_burgos_goc_nkua_chul

In [ ]:
df_result_burgos_goc_nkua = df_merged_burgos_goc_nkua.dropna()
df_result_burgos_goc_nkua

In [ ]:
df_result_burgos_goc_nkua_chul = df_merged_burgos_goc_nkua_chul.dropna()
df_result_burgos_goc_nkua_chul

## Selecting the best merged dataset of the above datasets for analysis

In [ ]:
drop_df_columns = ['dioptra_study', 'colon_time_point', 'tissue_sample_num']
df_result_burgos_goc_nkua_chul = df_result_burgos_goc_nkua_chul.drop(columns = drop_df_columns)
df_result_burgos_goc_nkua_chul

In [ ]:
df_result_burgos_goc_nkua_chul['dioptra_group_diagnosis'].value_counts()

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

custom_palette = {
    'Healthy': '#8B1E3F',               # Rich Burgundy
    'Non-Advanced Adenoma': '#B64040', # Warm Brick Red
    'Advanced Adenoma': '#6E2B62',     # Deep Plum
    'CRC': '#D18F50'                    # Warm Caramel
}

plt.figure(figsize = (12, 6))
sns.countplot(data = df_result_burgos_goc_nkua_chul, x = df_result_burgos_goc_nkua_chul['dioptra_group_diagnosis'].map(diagnosis_map), hue = df_result_burgos_goc_nkua_chul['dioptra_group_diagnosis'].map(diagnosis_map), order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = custom_palette)

plt.xlabel('The four groups of the diagnosis')
plt.ylabel('Number of patients')
plt.title('Distribution of dioptra_group_diagnosis')
plt.tight_layout()
# plt.savefig("diagnosis_distribution.png", dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
# The distribution of dioptra_group_diagnosis by sex
sex_map  = {0: "female", 1: "male"}
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

custom_palette = {
    'Healthy': '#8B1E3F',               # Rich Burgundy
    'Non-Advanced Adenoma': '#B64040', # Warm Brick Red
    'Advanced Adenoma': '#6E2B62',     # Deep Plum
    'CRC': '#D18F50'                    # Warm Caramel
}

sex_palette = {
    'female': '#8B1E3F',  # Burgundy
    'male': '#D18F50'     # Deep Plum
}

plt.figure(figsize = (12, 6))
sns.countplot(data = df_result_burgos_goc_nkua_chul, x = df_result_burgos_goc_nkua_chul['dioptra_group_diagnosis'].map(diagnosis_map), hue = df_result_burgos_goc_nkua_chul['sex'].map(sex_map), order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = sex_palette)
plt.title('Distribution of diagnosis by sex')
plt.xlabel('Diagnosis')
plt.ylabel('Number of patients')
# plt.xticks(ticks = [0, 1, 2, 3], labels = ['Healthy', 'Non-Advanced', 'Advanced', 'CRC']) # instead of diagnosis_map we can use this to set names on x-axis
plt.tight_layout()
#plt.savefig("diagnosis_distribution_by_sex.png", dpi = 400, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_result_burgos_goc_nkua_chul['age_colon_time_point'], bins=20, color="#B64040", edgecolor='black')
plt.axvline(df_result_burgos_goc_nkua_chul['age_colon_time_point'].median(), color='black', linestyle='--', label=f'Median: {df_result_burgos_goc_nkua_chul["age_colon_time_point"].median():.1f}')
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution')
plt.legend()
# plt.savefig("Age_Distribution_ds_12.png", dpi=400, bbox_inches="tight")
plt.show()

## Statistical Analysis

In [ ]:
# !pip install pingouin
import pingouin as pg
import scipy.stats as stats 
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.miscmodels.ordinal_model import OrderedModel

### Descriptive Statistics

In [ ]:
df = df_result_burgos_goc_nkua_chul.copy()
df

In [ ]:
df['dioptra_group_diagnosis'].value_counts()

In [ ]:
df['sex'].value_counts()

In [ ]:
df.describe()

In [ ]:
df.groupby('sex')['age_colon_time_point'].describe()

In [ ]:
categ_and_ordinal_columns = df.drop(columns = ['dioptra_id', 'age_colon_time_point', 'dioptra_group_diagnosis']).columns.tolist()

### Important functions for Statistical Analysis

#### Chi-square function for independency

Let's give some math details. First of all, this test is given by
$$ \Chi^2 = \frac{\sum_{i=1}^{r}\sum_{j=1}^{c}(O_{ij} - E_{ij})^2}{E_{ij}},  $$
where r is the number of the categories of the one column $X$ and c is the number of the categories of the other $Y$ (for us, this is the categories of the label, i.e., 4). Also, the quantity $ O_{ij} $ is the value of the $ (i,j) $ position of the contigency table and the quantity $ E_{ij} $ is given by 
$$ E_{ij} = \frac{\sum_{i=1}^r O_{ij}\sum_{j=1}^c O_{ij}}{N}, $$
where N is the number of the rows of the DataFrame (i.e. 336).
So, the null hypothesis is
$$ H_0 : X \ \text{and} \ \text{Y are independent} $$
and the alternative hypothesis is
$$ H_{al} : X \ \text{and} \ \text{Y are dependent}. $$
Therefore, the null hypothesis is rejected with level of significance $a$ (usually $a=0.05$), if
$$ \Chi^2\geq \Chi^2_{(r-1)(c-1),a} $$
or if the $p$-value is less than $a$, i.e.,
$$ p-\text{value} < a $$

* Important remark (see Batsidis' notes - file 3 - page 54): This test is applicable if a) the total number of our data sample is quadruple the dimension of the contigency table and  b) the expected frequencies $E_{ij}$ is not less than 1 and the 25% of them are not less than 5

In [ ]:
def chi_square_for_dependency(df, feature_list, target_feature):
    """
    Performs Chi-square tests between each feature in feature_list and a target categorical variable
    """
    
    chi2_valid_list = []
    chi2_not_valid_list = []
    expect_list_for_not_valid = []
    dependent_features = []

    for col in feature_list:

        # Crosstabulation (contingency table)
        cont = pd.crosstab(df[col], df[target_feature])

        # Chi-square test
        chi2, p, dof, expect_freq = stats.chi2_contingency(cont)

        # Chi-square validity conditions
        valid = (
            df.shape[0] > 4 * cont.size and       # sample size condition
            (expect_freq >= 1).all() and          # each expected freq >= 1
            (expect_freq < 5).mean() <= 0.25      # at most 25% < 5
        )

        if valid:
            print(f"VALID Chi-square for '{col}' vs '{target_feature}'")
            print(f" χ² = {chi2:.3f},  p = {p:.3f},  dof = {dof}")

            if p < 0.05:
                print(f" Reject H₀ : '{col}' is DEPENDENT on '{target_feature}'")
                dependent_features.append(col)
            else:
                print(f" Fail to reject H₀ : No evidence of dependency between {col} and {target_feature}")

            chi2_valid_list.append(col)

        else:
            print(f"NOT VALID Chi-square for '{col}' vs '{target_feature}'")
            chi2_not_valid_list.append(col)
            expect_list_for_not_valid.append((col, expect_freq))

        print("-" * 80)

    return {
        "valid": chi2_valid_list,
        "not_valid": chi2_not_valid_list,
        "invalid_expected_freq": expect_list_for_not_valid,
        "dependent": dependent_features
    }

#### ASR (Adjusted standardized residuals) function. Be focused on values |ASR| > 1.96

* If for example we have a value 3.5 on a cell (1x1), then this value says that the records which belongs to row 1 and column 1 are more than expected!
* and if we have negative value -3.5, then this value says that that cell is less than expected!

In [ ]:
# the adjusted_standardized_residuals as defined in Batsidis'notes, page 55
def adjusted_standardized_residuals(table):
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    row_totals = table.sum(axis=1).values.reshape(-1, 1)
    col_totals = table.sum(axis=0).values.reshape(1, -1)
    O, E = table.values, expected
    ASR = (O - E) / (np.sqrt(E) * (np.sqrt((1 - row_totals / n) * (1 - col_totals / n))))

    return pd.DataFrame(ASR, index = table.index, columns = table.columns)

#### Outlier percentage function

In [ ]:
def outlier_percentage(series):
    """Return the percentage of outliers in a pandas Series using IQR (interquartile range) rule."""
    q1, q3 = np.percentile(series, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = ((series < lower) | (series > upper)).sum()
    return 100 * outliers / len(series)

#### Shapiro wilk test function with QQ and histogram plots

In [ ]:
def shapiro_test_by_group_QQ_and_hist_plots(df, group_list, numeric_list):
    for group in group_list:
        for num_var in numeric_list:

            values = df.loc[df['dioptra_group_diagnosis'] == group, num_var]
            values = values.dropna()

            # Shapiro–Wilk test
            stat, p = stats.shapiro(values)

            print(f"length of the group {group} = {len(values)}, W={stat:.3f}, p={p:.3f}")

            if p > 0.05:
                print(f"The H_0 is not rejected, so we can say that the group {group} can be approximated by a Normal Distribution\n")
            else:
                print(f"The H_0 is rejected, so we can say that the group {group} cannot be approximated by a Normal Distribution\n")

            # PLOTS
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))

            stats.probplot(values, dist="norm", plot=axes[0])
            axes[0].set_title(f"Q–Q plot of {num_var} for group {group}")

            sns.histplot(x=values, ax=axes[1])
            axes[1].set_title(f"Histogram of {num_var} for group {group}")

            plt.tight_layout()
            plt.show()

            print("-" * 130)


#### Brant test function in order to check the validity of the Ordinal Logistic Regression (OLR)
If p-value of a feature is less than 0.05 this means the effect of the predictor is not constant across categories of your ordinal outcome, so the proportional odds assumption is not satisfied. For more details se the sources from Medium. For the first one click [here](https://ujangriswanto08.medium.com/step-by-step-guide-to-understanding-ordinal-logistic-regression-e7477e846914) and for the second on click [here](https://medium.com/evangelinelee/brant-test-for-proportional-odds-in-r-b0b373a93aa2).

If p-value > 0.05 then the proportional odds assumption is satisfied and the predictor is ok!
* If we want valid results for the OLR, we need all the features of the DataFrame having p-value > 0.05

In [ ]:
def brant_test(df, y, X):
    # Fit ordered logit (proportional odds)
    mod = OrderedModel(df[y], df[X], distr='logit')
    res = mod.fit(method='bfgs', disp=False)

    # Extract outcome levels
    levels = np.sort(df[y].unique())
    k = len(levels)

    # Store binary logit results
    logit_params = []
    logit_cov = []

    for j in range(1, k):
        # Create binary split: Y >= j+1
        y_bin = (df[y] >= levels[j]).astype(int)
        logit = sm.Logit(y_bin, sm.add_constant(df[X])).fit(disp=False)
        logit_params.append(logit.params.values)
        logit_cov.append(logit.cov_params().values)

    # Test proportionality for each predictor
    results = {}
    num_splits = k - 1
    num_vars = len(X) + 1  # predictors + intercept

    for i, var in enumerate(["const"] + X):
        # Stack coefficients for this variable
        coeffs = np.array([logit_params[j][i] for j in range(num_splits)])
        mean_coeff = np.mean(coeffs)

        # Differences from mean
        diffs = coeffs - mean_coeff

        # Variance-covariance block diagonal
        cov_blocks = np.zeros((num_splits, num_splits))
        for j in range(num_splits):
            cov_blocks[j, j] = logit_cov[j][i, i]

        # Chi-square test statistic
        stat = diffs.T @ np.linalg.inv(cov_blocks) @ diffs
        pval = 1 - stats.chi2.cdf(stat, df=num_splits - 1)

        results[var] = (stat, pval)

    return pd.DataFrame(results, index=["Chi2", "p-value"])

#### Ordinal Logistic Regression (OLR) function

In [ ]:

def ordinal_logistic_regression(X, y, distr = 'logit', method = 'bfgs', maxiter = None):  # logit = logistic cumulative distribution function as the link for the ordinal thresholds, 
                                                                      # BFGS is a numerical optimization algorithm used to find the maximum likelihood estimates (gradient-based optimization). You can use also a deterministic optimizer, i.e., Newton
    """                                                               
    Fits an Ordinal Logistic Regression model using statsmodel OrderedModel.
    """

    # Build model
    model = OrderedModel(y, X, distr=distr)

    # Fit model
    if maxiter is not None:
        result = model.fit(method=method, maxiter=maxiter) # This is only for the Newton-Rampson method. The default maxiter=100
    else:
        result = model.fit(method=method)
    
    # Print summary
    print("\n Model Summary:\n")
    print(result.summary())

    # Extract coefficients, p-values, confidence intervals
    params = result.params.loc[X.columns]
    pvals = result.pvalues.loc[X.columns]
    conf = result.conf_int().loc[X.columns]

    # Construct OR table
    needed_params_df = pd.DataFrame({
        'coef': params,
        'p-value': pvals,
        'OR': np.exp(params),
        'OR_2.5%': np.exp(conf[0]),
        'OR_97.5%': np.exp(conf[1])
    })

    # Sort predictors by p-value
    needed_params_df = needed_params_df.sort_values('p-value')

    return needed_params_df


#### Partial Proportional Odds Model (PPOM) function

If the proportional odds assumption is not satisfied for each predictor feature, then we need to use PPOM to distinguish lets say the 'bad' and the 'good' features provided by the Brant test. The parallel_vars list contains the 'good' features while the nonparallel_vars contains the 'bad' features.

In [ ]:
def partial_proportional_odds_model(df, y_col, parallel_vars, nonparallel_vars):
    # Ensure y is ordinal-coded with increasing order
    levels = np.sort(df[y_col].dropna().unique())
    K = len(levels)

    # Create an ID for clustering (one subject -> multiple stacked rows)
    df0 = df.copy().reset_index(drop=True)
    df0["_id"] = np.arange(len(df0))

    rows = []
    # We create J = K-1 cumulative splits
    # Here we use: y_bin = 1{ y >= levels[j] } for j=1..K-1
    for j in range(1, K):
        thr = levels[j]
        tmp = df0[["_id", y_col] + parallel_vars + nonparallel_vars].copy()
        tmp["_thr"] = j  # threshold index
        tmp["y_bin"] = (tmp[y_col] >= thr).astype(int)
        rows.append(tmp)

    long = pd.concat(rows, ignore_index=True)

    # Threshold dummies => different intercepts per threshold (no global intercept)
    thr_dum = pd.get_dummies(long["_thr"], prefix="thr", drop_first=False)

    # Parallel part: same slope across thresholds
    X_par = long[parallel_vars].copy()

    # Non-parallel part: allow slopes vary by threshold via interactions
    # Create interactions: x * thr_dummy for each threshold dummy
    X_np = []
    for v in nonparallel_vars:
        inter = thr_dum.mul(long[v], axis=0)
        inter.columns = [f"{v}:{c}" for c in inter.columns]
        X_np.append(inter)
    X_np = pd.concat(X_np, axis=1) if len(X_np) else pd.DataFrame(index=long.index)

    # Combine design matrix: threshold intercepts + parallel + nonparallel interactions
    X = pd.concat([thr_dum, X_par, X_np], axis=1)

    # after thr_dum calculations the X dataframe contains boolean variables which is not permitted belo. So, we force all the variables to be float before fitting the model
    X = X.astype(float)


    # Fit logistic regression (Binomial GLM), cluster-robust SE by original id
    model = sm.GLM(long["y_bin"], X, family=sm.families.Binomial())
    res = model.fit(cov_type="cluster", cov_kwds={"groups": long["_id"]})
    
    print(res.summary())

    return res, levels


#### PPOM function containing the most important information in order to obtain the risk and protective factors

In [ ]:
def ppom_or_table(res, levels, parallel_vars, nonparallel_vars, alpha=0.05, thr_labels = None):
  
    params = res.params
    pvals = res.pvalues
    conf = res.conf_int()

    out = []
    K = len(levels)

    # j=1 => Y >= level[1] vs < level[1], etc.
    if thr_labels is None:
        thr_labels = {j: f"P(Y ≥ {levels[j]}) vs P(Y < {levels[j]})" for j in range(1, K)}

    # parallel variables: one row each
    for v in parallel_vars:
        if v in params.index:
            out.append({
                "variable": v,
                "type": "parallel",
                "threshold": "all (common effect)",
                "coef": params[v],
                "p_value": pvals[v],
                "OR": np.exp(params[v]),
                "OR_2.5%": np.exp(conf.loc[v, 0]),
                "OR_97.5%": np.exp(conf.loc[v, 1]),
                "significant": pvals[v] < alpha
            })

    # non-parallel variables: one row per threshold dummy interaction
    for v in nonparallel_vars:
        for j in range(1, K):
            name = f"{v}:thr_{j}"
            if name in params.index:
                out.append({
                    "variable": v,
                    "type": "nonparallel",
                    "threshold": thr_labels[j],
                    "coef": params[name],
                    "p_value": pvals[name],
                    "OR": np.exp(params[name]),
                    "OR_2.5%": np.exp(conf.loc[name, 0]),
                    "OR_97.5%": np.exp(conf.loc[name, 1]),
                    "significant": pvals[name] < alpha
                })

    df_out_ppom = pd.DataFrame(out)

    # Sort by p-value (like your OLR table)
    df_out_ppom = df_out_ppom.sort_values("p_value").reset_index(drop=True)

    return df_out_ppom


#### Variance Inflation Factor (VIF) function

In [ ]:
def compute_vif(X):
    vif_df = pd.DataFrame()
    vif_df['variable'] = X.columns
    vif_df['VIF'] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    vif_df = vif_df.sort_values('VIF', ascending=False).reset_index(drop=True)

    return vif_df

### Data visualization and EDA

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}
counts = df['dioptra_group_diagnosis'].map(diagnosis_map).value_counts()
counts

In [ ]:
percentages = counts / counts.sum() * 100
percentages

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

custom_palette = {
    'Healthy': '#8B1E3F',               # Rich Burgundy
    'Non-Advanced Adenoma': '#B64040', # Warm Brick Red
    'Advanced Adenoma': '#6E2B62',     # Deep Plum
    'CRC': '#D18F50'                    # Warm Caramel
}

plt.figure(figsize = (12, 6))
sns.countplot(data = df, x = df['dioptra_group_diagnosis'].map(diagnosis_map), hue = df['dioptra_group_diagnosis'].map(diagnosis_map), legend=False, order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = custom_palette)

plt.xlabel('')
plt.ylabel('')
plt.title('Distribution of diagnosis', fontsize = 16, fontweight='bold')
plt.tight_layout()
plt.xticks(fontsize=14, fontweight='bold')

# plt.savefig("diagnosis_distribution.png", dpi = 400, bbox_inches = 'tight')
plt.show()

In [ ]:
# The distribution of the dioptra_group_diagnosis grouped by all the 'categ_and_ordinal_columns'
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

for col in categ_and_ordinal_columns:
    plt.figure(figsize = (12, 6))
    sns.countplot(data = df, x = df['dioptra_group_diagnosis'].map(diagnosis_map), hue = col, order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], palette = 'tab10')
    plt.title(f"Distribution of dioptra_group_diagnosis grouped by {col}")
    plt.xlabel('Diagnosis')
    plt.ylabel('Total records')
    # plt.xticks(ticks = [0, 1, 2, 3], labels = ['Healthy', 'Non-Advanced', 'Advanced', 'CRC']) # instead of diagnosis_map we can use this to set names on x-axis
    plt.tight_layout()
    plt.show()

In [ ]:
# plot for presentation
cardiovascular_palette = {
    'No': '#8B1E3F',  # Burgundy (No)
    'Yes': '#D18F50'   # Warm Caramel (Yes)
}

cardiovascular_map = {0: 'No', 1: 'Yes'}
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

plt.figure(figsize = (12, 6))
sns.countplot(data = df, x = df['dioptra_group_diagnosis'].map(diagnosis_map), hue = df['disease_cardiovascular'].map(cardiovascular_map), order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'], hue_order = ['No', 'Yes'], palette = cardiovascular_palette)
plt.title(f"Distribution of dioptra_group_diagnosis grouped by disease_cardiovascular")
plt.xlabel('Diagnosis')
plt.ylabel('Total records')
# plt.xticks(ticks = [0, 1, 2, 3], labels = ['Healthy', 'Non-Advanced', 'Advanced', 'CRC']) # instead of diagnosis_map we can use this to set names on x-axis
plt.tight_layout()
# plt.savefig("disease_cardiovascular_distribution_ds_12.png", dpi = 400, bbox_inches = 'tight')
plt.show()

In [ ]:
# distribution of sex grouped by dioptra_group_diagnosis
sex_map = {0: 'Female', 1: 'Male'}

plt.figure(figsize = (12, 6))
sns.countplot(data = df, x = df['sex'].map(sex_map), hue = 'dioptra_group_diagnosis', order = ['Female', 'Male'], palette = 'tab10')
plt.title(f"Distribution of sex")
plt.xlabel('Sex')
plt.ylabel('Total records')
plt.tight_layout()
plt.show()

In [ ]:
# the numbers of the above plot
df.groupby('sex')['dioptra_group_diagnosis'].value_counts()

In [ ]:
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

custom_palette = {
    'Healthy': '#8B1E3F',               # Rich Burgundy
    'Non-Advanced Adenoma': '#B64040', # Warm Brick Red
    'Advanced Adenoma': '#6E2B62',     # Deep Plum
    'CRC': '#D18F50'                    # Warm Caramel
}

fig, axes = plt.subplots(1, 2, figsize = (16, 5))
# histogram of age_colon_time_point for each dioptra_group_diagnosis
sns.histplot(data = df, x = 'age_colon_time_point', bins = 30, kde = True, hue = df['dioptra_group_diagnosis'].map(diagnosis_map), palette = custom_palette, ax = axes[0])
axes[0].set_xlabel('age_colon_time_point')

# probability density function of age_colon_time_point for each dioptra_group_diagnosis 
sns.kdeplot(data = df, x = 'age_colon_time_point', hue = df['dioptra_group_diagnosis'].map(diagnosis_map), palette = custom_palette, ax = axes[1])
axes[1].set_xlabel('age_colon_time_point')

plt.tight_layout()
# plt.savefig("density_and_histogram_ages.png", dpi = 300, bbox_inches = 'tight')
plt.show()

### Univariant data analysis for categorical and ordinal columns

We have to compare categorical or ordinal variables VS diagnosis which is an ordinal variable, so we will use the Chi^2 statistical test. So, let's give some math details. First of all, this test is given by
$$ \Chi^2 = \frac{\sum_{i=1}^{r}\sum_{j=1}^{c}(O_{ij} - E_{ij})^2}{E_{ij}},  $$
where r is the number of the categories of the one column $X$ and c is the number of the categories of the other $Y$ (for us, this is the categories of the label, i.e., 4). Also, the quantity $ O_{ij} $ is the value of the $ (i,j) $ position of the contigency table and the quantity $ E_{ij} $ is given by 
$$ E_{ij} = \frac{\sum_{i=1}^r O_{ij}\sum_{j=1}^c O_{ij}}{N}, $$
where N is the number of the rows of the DataFrame (i.e. 336).
So, the null hypothesis is
$$ H_0 : X \ \text{and} \ \text{Y are independent} $$
and the alternative hypothesis is
$$ H_{al} : X \ \text{and} \ \text{Y are dependent}. $$
Therefore, the null hypothesis is rejected with level of significance $a$ (usually $a=0.05$), if
$$ \Chi^2\geq \Chi^2_{(r-1)(c-1),a} $$
or if the $p$-value is less than $a$, i.e.,
$$ p-\text{value} < a $$

* Important remark (see Batsidis' notes - file 3 - page 54): This test is applicable if a) the total number of our data sample is quadruple the dimension of the contigency table and  b) the expected frequencies $E_{ij}$ is not less than 1 and the 25% of them are not less than 5

In [ ]:
chi_square_relusts = chi_square_for_dependency(df = df, feature_list = categ_and_ordinal_columns, target_feature = 'dioptra_group_diagnosis')

In [ ]:
print(f'The features whose the chi_square test is NOT VALID are\n {chi_square_relusts["not_valid"]}')

In [ ]:
print(f'The dependent features with the dioptra_group_diagnosis are \n{chi_square_relusts["dependent"]}')

In [ ]:
# calculate the adjusted standardized residuals for the dependent columns included on chi_square_relusts["dependent"] list
for depend_elem in chi_square_relusts["dependent"]:
    cont_table = pd.crosstab(df[depend_elem], df['dioptra_group_diagnosis'])
    asr = adjusted_standardized_residuals(table = cont_table)
    
    # create the asr as plots for better understanding
    plt.figure(figsize=(8,4))
    sns.heatmap(asr, annot = True, cmap = "coolwarm", fmt = ".2f")
    plt.title(f"Adjusted Standardized Residuals ({depend_elem} × Dioptra Group)")
    plt.show()

### Univariant data analysis for the numeric column age_colon_time_point

In [ ]:
df.boxplot(column = 'age_colon_time_point', by = 'dioptra_group_diagnosis');

In [ ]:
# We confirm here the above boxplots!

# check per group
for g in sorted(df['dioptra_group_diagnosis'].unique()):
    vals = df.loc[df['dioptra_group_diagnosis'] == g, 'age_colon_time_point']
    perc = outlier_percentage(vals)
    print(f"Group {g}: {perc:.1f}% outliers (n={len(vals)})")

* The outlier rule is confirmed (each group has less than 10% missing values)

In [ ]:
shapiro_test_by_group_QQ_and_hist_plots(df=df, group_list = df['dioptra_group_diagnosis'].unique(), numeric_list = ['age_colon_time_point'])

Shapiro-Wilk tests indicates that both of the 4 groups cannon be approximated by a Normal Distribution. It is an expected result because these tests (such as Shapiro-Wilk, Kolmogorov-Smirnov, etc) are sensitive when the length of the group getting higher and higher. But, since each group has length greater than 50 we can continue our analysis with parametric tests in order to take our results approximately through the Central Limit Theorem

In [ ]:
# Levene test and Brown-Forsythe test for the equality hypothesis of the variances
# Gather groups
groups = [df.loc[df['dioptra_group_diagnosis'] == g, 'age_colon_time_point'] for g in sorted(df['dioptra_group_diagnosis'].unique())]

# Variance check
# Levene (center='mean' is classical Levene; center='median' -> Brown-Forsythe)
levene_stat, levene_p = stats.levene(*groups, center = 'mean')
bf_stat, bf_p = stats.levene(*groups, center = 'median')  # Brown-Forsythe via Levene with median

print(f"Levene (mean) p = {levene_p:.3f}")
print(f"Brown-Forsythe (median) p = {bf_p:.4f}")

As we see, the Levene and Brown-Forsythe tests have p-values < 0.05 so the equality hypothesis of the variances are rejected and we need to check the hypothesis $H_0 : μ_1 = μ_2 = μ_3 = μ_4$ using the Welch test

In [ ]:
# The Welch test for the equality of the means
welch_res = pg.welch_anova(dv = 'age_colon_time_point', between = 'dioptra_group_diagnosis', data = df)
print(welch_res)

 Since, the p-value (Welch) = 3.067010e-99 < 0.05 we reject the null hypothesis, i.e. $H_0: μ_1 = μ_2 = μ_3 = μ_4$, so we have evidence to say tha the 'enrollmennt_age' differs between the four diagnostic groups. In other words, the column 'age' is dependent on the label 'dioptra_group_diagnosis'.
 

Since we have rejected the null hypothesis of equal means, meaning there is a significant difference somewhere among these four groups, we want to know which specific groups differ.
We wil use multiple comparisons test which are in general the correct choice depends on whether the group variances are equal or not (see batsidis' notes, page 162)

In [ ]:
# for the column 'age_colon_time_point' the null hypothesis of equal variances is rejected, so we use Games-Howell which is a non-parametric test for the multiple comparisons
games_howell_age = pg.pairwise_gameshowell(
    dv = 'age_colon_time_point', between = 'dioptra_group_diagnosis', data = df
)
games_howell_age

In [ ]:
# cumulative plots 
plt.figure(figsize=(12,6))
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

custom_palette = {
    'Healthy': '#8B1E3F',               # Rich Burgundy
    'Non-Advanced Adenoma': '#B64040', # Warm Brick Red
    'Advanced Adenoma': '#6E2B62',     # Deep Plum
    'CRC': '#D18F50'                    # Warm Caramel
}

sns.boxplot(x = df['dioptra_group_diagnosis'].map(diagnosis_map), y = 'age_colon_time_point', data = df, order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'],palette=custom_palette )  #palette = custom_palette
sns.stripplot(x = df['dioptra_group_diagnosis'].map(diagnosis_map), y = 'age_colon_time_point', data = df, color = '#808080', alpha = 0.5)
plt.title("Age vs Diagnosis Group (Games–Howell post hoc analysis)")
plt.xlabel("Diagnosis")
plt.ylabel("Age")
# plt.savefig("Age_VS_diagnosis_post_hoc_ds_12.png", dpi = 400, bbox_inches = 'tight')
plt.show()


1) So, based on the above results taken from the Games-Howell test, we can say that CRC group has much higher mean age that Healthy group.
2) Furthermore, mean age increases steadily from group 1 to 2 to 3 to 4!

* Let us fix the normality for each group and see if we have the same results fot the age

In [ ]:
df_boxcox = df.copy()
age_boxcox, lambda_boxcox = stats.boxcox(df_boxcox['age_colon_time_point'].values)
df_boxcox['age_boxcox'] = age_boxcox
df_boxcox

In [ ]:
df_boxcox.boxplot(column = 'age_boxcox', by = 'dioptra_group_diagnosis');

In [ ]:
# We confirm here the above boxplots
for g in sorted(df_boxcox['dioptra_group_diagnosis'].unique()):
    vals = df_boxcox.loc[df_boxcox['dioptra_group_diagnosis'] == g, 'age_boxcox']
    perc = outlier_percentage(vals)
    print(f"Group {g}: {perc:.1f}% outliers (n={len(vals)})")

The outlier rule is confirmed (each group has less than 10% missing values)

In [ ]:
shapiro_test_by_group_QQ_and_hist_plots(df = df_boxcox, group_list = df['dioptra_group_diagnosis'].unique(), numeric_list = ['age_boxcox'])

The test does not give us the normality but the graphs are much closer to a Normal Distribution

In [ ]:
# Levene test and Brown-Forsythe test for the equality hypothesis of the variances
# Gather groups_box
groups_box = [df_boxcox.loc[df_boxcox['dioptra_group_diagnosis'] == g, 'age_boxcox'] for g in sorted(df_boxcox['dioptra_group_diagnosis'].unique())]

# Variance check
# Levene (center='mean' is classical Levene; center='median' -> Brown-Forsythe)
levene_stat2, levene_p2 = stats.levene(*groups_box, center = 'mean')
bf_stat2, bf_p2 = stats.levene(*groups_box, center = 'median')  # Brown-Forsythe via Levene with median

print(f"Levene (mean) p = {levene_p2:.3f}")
print(f"Brown-Forsythe (median) p = {bf_p2:.4f}")

As we see, the Levene and Brown-Forsythe tests have p-values < 0.05 so the equality hypothesis of the variances are rejected and we need to check the hypothesis $H_0 : μ_1 = μ_2 = μ_3 = μ_4$ using the Welch test

In [ ]:
# The Welch test for the equality of the means
welch_res = pg.welch_anova(dv = 'age_boxcox', between = 'dioptra_group_diagnosis', data = df_boxcox)
print(welch_res)

 Since, the p-value (Welch) = 2.325629e-97 < 0.05 we reject the null hypothesis, i.e. $H_0: μ_1 = μ_2 = μ_3 = μ_4$, so we have evidence to say tha the 'enrollmennt_age' differs between the four diagnostic groups. In other words, the column 'age' is dependent on the label 'dioptra_group_diagnosis'.
 

Since we have rejected the null hypothesis of equal means, meaning there is a significant difference somewhere among these four groups, we want to know which specific groups differ.
We wil use multiple comparisons test which are in general the correct choice depends on whether the group variances are equal or not (see batsidis' notes, page 162)

In [ ]:
# for the column 'age_boxcox' the null hypothesis of equal variances is rejected, so we use Games-Howell which is a non-parametric test for the multiple comparisons
games_howell_age = pg.pairwise_gameshowell(
    dv = 'age_boxcox', between = 'dioptra_group_diagnosis', data = df_boxcox
)
games_howell_age

In [ ]:
# cumulative plots 
plt.figure(figsize=(12,6))
diagnosis_map = {1: 'Healthy', 2: 'Non-Advanced Adenoma', 3: 'Advanced Adenoma', 4: 'CRC'}

custom_palette = {
    'Healthy': '#8B1E3F',               # Rich Burgundy
    'Non-Advanced Adenoma': '#B64040', # Warm Brick Red
    'Advanced Adenoma': '#6E2B62',     # Deep Plum
    'CRC': '#D18F50'                    # Warm Caramel
}

sns.boxplot(x = df_boxcox['dioptra_group_diagnosis'].map(diagnosis_map), y = 'age_boxcox', data = df_boxcox, order = ['Healthy', 'Non-Advanced Adenoma', 'Advanced Adenoma',  'CRC'],palette=custom_palette )  #palette = custom_palette
sns.stripplot(x = df_boxcox['dioptra_group_diagnosis'].map(diagnosis_map), y = 'age_boxcox', data = df_boxcox, color = '#808080', alpha = 0.5)
plt.title("Age_boxcox vs Diagnosis Group (Games–Howell post hoc analysis)")
plt.xlabel("Diagnosis")
plt.ylabel("Age_boxcox")
# plt.savefig("Age_VS_diagnosis_post_hoc.png", dpi = 300, bbox_inches = 'tight')
plt.show()


We have the same results as before:
1) So, based on the above results taken from the Games-Howell test, we can say that CRC group has much higher mean age that Healthy group.
2) Furthermore, mean age increases steadily from group 1 to 2 to 3 to 4!

### Multivariant data analysis

We will follow the workflow:
1) Encode all the independent features (68 columns) because the method is based on regression models. Keep the binary columns as it is. For a k categorical (nominal) column create k-1 dummy variables columns. For the ordinal columns keep one column but be careful on the ordering. Keep in mind that the OLR uses only rank ordering and not numeric distance! For example, the orderings 1 < 2 < 3 and 1 < 2 < 4 are exactly the same for the OLR.

2) Compute VIF (Variance Inflation Factor) for multicolinearity in order to detect dependent features.

3) Remove or merge collinear variables. For example instead of having both diabetes and insulin, keep only the suitable one or merge them.

4) Fit the multivariant suitable test Ordinal Logistic Regression (OLR) and be focused on the odds ratios (OR) given by $OR_i = e^{\beta_i}$, where $\beta_i$ are the coefficients of the feature (predictor) $X_i$ of the OLR regression model.


In [ ]:
df

In [ ]:
columns_to_drop_multi = ['dioptra_id']
nominal_columns_multi = ['smoking', 'allergies_asthma']

In [ ]:
# drop unnecessary columns
df_multi = df.drop(columns = columns_to_drop_multi)

# convert the nominal columns to integer type (Int64). For example, we convert the smokin_1.0 to smoking_1. This is an important change here for the PPOM function
df_multi[nominal_columns_multi] = df_multi[nominal_columns_multi].astype("Int64")

# convert the nominal variables into to dummy variables , droping the first value of each dummy variable avoiding multicollinearity (dummy variable trap)
df_multi = pd.get_dummies(df_multi, columns = nominal_columns_multi, drop_first = True, dtype = int)

# print df_multi
df_multi

In [ ]:
# the input features (independent variables) of the dataframe df_multi
X = df_multi.drop(columns = ['dioptra_group_diagnosis'])

# the target feature (dependent variable)
y = df_multi['dioptra_group_diagnosis']

Now before we continue we need to check if we can trust the ordinal logistic regresion. So we need to use the Brant test. 

In [ ]:
brant_results = brant_test(df_multi, y = 'dioptra_group_diagnosis', X = X.columns.to_list()) # X must be a list 
brant_results.T

In [ ]:
# Compute VIF for all input features
vif_df = compute_vif(X=X)
vif_df

#### The Ordinal Logistic Regresion approach (not the best here since the proportional odds assumption is rejected)

In [ ]:
olr_df = ordinal_logistic_regression(X = X, y = y, distr = 'logit', method='bfgs')

In [ ]:
olr_df

#### The Partial Proportional Odds approach (the best here)

In [ ]:
non_parallel_list = [
    "age_colon_time_point", "sex", "drinking_behavior", "dyslipidemia",
    "disease_cardiovascular", "allergy_drug", "antihypertensives", "smoking_2"
]
parallel_list = [
    "nsaid", "diabetes_mellitus_2", "hypertension", "disease_chronic_kidney",
    "antiocoagulants", "aspirin", "statin", "insulin", "glp_1", "sglt_2",
    "metaformin", "antiplatelet", "corticosteroids", "dpp_4",
    "smoking_1", "allergies_asthma_1", "allergies_asthma_2", "allergies_asthma_3"
]

In [ ]:
res, levels = partial_proportional_odds_model(df = df_multi, y_col = 'dioptra_group_diagnosis', parallel_vars = parallel_list, nonparallel_vars = non_parallel_list)

In [ ]:
ppom_table = ppom_or_table(res, levels, parallel_list, non_parallel_list, alpha=0.05, thr_labels={1: "≥ non-advanced adenoma vs healthy", 2: "≥ advanced adenoma vs ≤ non-advanced", 3: "CRC vs ≤ advanced adenoma"})
ppom_table

From the above `ppom_table` we obtain the risk and protective factors!